<a href="https://colab.research.google.com/github/BernardoBremer/Inteligencia-Computacional-Cetys-/blob/main/9_vectorstore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vector stores and semantic search



## Part I: Basic vector store implementation

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = None

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        new_embeddings = self.embedding_model.encode(texts, show_progress_bar=True)
        self.documents.extend(documents)
        if self.embeddings is None:
            self.embeddings = np.array(new_embeddings)
        else:
            self.embeddings = np.vstack([self.embeddings, np.array(new_embeddings)])

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        query_embedding = self.embedding_model.encode([query])[0]
        norms = np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(query_embedding)
        similarities = np.dot(self.embeddings, query_embedding) / norms
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [SearchResult(score=float(similarities[i]), document=self.documents[i]) for i in top_indices]

In [6]:
df = pd.read_csv("animal-fun-facts-dataset.csv")
df = df.fillna("")

documents = []
for _, row in df.iterrows():
    documents.append(Document(
        text=str(row["text"]),
        metadata={
            "animal_name": str(row["animal_name"]),
            "source": str(row["source"]),
            "media_link": str(row["media_link"]),
            "wikipedia_link": str(row["wikipedia_link"])
        }
    ))

print(f"Documentos cargados: {len(documents)}")

store = VectorStore(model)
store.add_documents(documents)

Documentos cargados: 7734


Batches:   0%|          | 0/242 [00:00<?, ?it/s]

In [9]:
def mostrar_resultados(query, results):
    print(f"Consulta: '{query}'")
    print()
    for i, r in enumerate(results):
        print(f"  Resultado {i+1}:")
        print(f"    Score: {r.score:.4f}")
        print(f"    Texto: {r.document.text[:200]}")
        print(f"    Metadatos: {r.document.metadata}")
        print()

consultas = [
    "animals that can fly very fast",
    "venomous snakes and their dangerous bite",
    "deep sea ocean creatures bioluminescence",
    "largest animals in the world by weight",
    "endangered species and conservation efforts"
]

for q in consultas:
    results = store.search(q, top_k=3)
    mostrar_resultados(q, results)
    print()

Consulta: 'animals that can fly very fast'

  Resultado 1:
    Score: 0.7108
    Texto: Fastest animal on Earth
    Metadatos: {'animal_name': 'peregrine falcon', 'source': 'https://a-z-animals.com/animals/peregrine-falcon/', 'media_link': '', 'wikipedia_link': '/wiki/Peregrine_falcon'}

  Resultado 2:
    Score: 0.6832
    Texto: The fastest creatures on the planet!
    Metadatos: {'animal_name': 'falcon', 'source': 'https://a-z-animals.com/animals/falcon/', 'media_link': '', 'wikipedia_link': '/wiki/Falcon'}

  Resultado 3:
    Score: 0.6659
    Texto: They are the fastest bird in the world.
The peregrine falcon’s streamlined body and pointed wings allow it to reach high speeds in flight and when diving to hunt prey.
    Metadatos: {'animal_name': 'peregrine falcon', 'source': 'https://factanimal.com/peregrine-falcon/', 'media_link': '', 'wikipedia_link': '/wiki/Peregrine_falcon'}


Consulta: 'venomous snakes and their dangerous bite'

  Resultado 1:
    Score: 0.8487
    Texto: The 

## Part II: Filtering by metadata

In [8]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass